# OASIS-1: Fixed slice vs Canny-selected slice — controlled CNN comparison

**Binary outcome:** CDR=0 vs CDR>0 (dementia classification, **not** an AD-specific diagnosis). Uses original OASIS-1 T1 MRI `.img`+`.hdr` volumes and cross-sectional metadata. One 2D slice per subject per method, with **exactly the same subjects, CNN architecture, splits and hyperparameters**.

**Data leakage precautions:** subject-level split before training; only training images augment; no labels used to choose Canny slices; test set evaluated once after independent validation-based checkpoint choice. Age≥60 by default. If scans per person duplicate, one T88 scan is selected deterministically.

**Before running:** put the OASIS-1 disc archives and `oasis_cross-sectional.xlsx` (or `.csv`) in `DATA_DIR` (set in section 1; locally `~/Documents/phm5005 Testset/data/OASIS1_PROJECT`, on Colab `MyDrive/OASIS1_PROJECT`). The notebook extracts them automatically and runs both locally and on Colab.

## 1. Dependencies and configuration

In [ ]:
# Works locally (VS Code/Jupyter) and on Colab. Installs only what is missing.
%pip -q install nibabel openpyxl scikit-learn opencv-python-headless torch torchvision pandas matplotlib pillow

In [ ]:
import sys, zipfile, tarfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/OASIS1_PROJECT")   # disc archives + metadata
    MRI_ROOT = Path("/content/OASIS1")                          # extraction target
    WORK_DIR = Path("/content")
else:
    # >>> LOCAL: put the OASIS-1 disc archives and oasis_cross-sectional.xlsx/.csv here <<<
    PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = PROJ / "data" / "raw"
    MRI_ROOT = PROJ / "data" / "extracted"
    WORK_DIR = PROJ / "results" / "notebook_outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True); MRI_ROOT.mkdir(parents=True, exist_ok=True); WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Colab:", IN_COLAB, "| DATA_DIR:", DATA_DIR, "| MRI_ROOT:", MRI_ROOT)

def _safe_tar_extract(t, dest):
    try:
        t.extractall(dest, filter="data")      # Python >= 3.10.12 / 3.11.4 / 3.12
    except TypeError:
        t.extractall(dest)

if any(MRI_ROOT.rglob("*t88_masked_gfc.img")):
    print("T88 volumes already present - skipping extraction.")
else:
    archives = sorted(p for p in DATA_DIR.glob("*disc*")
                      if p.name.lower().endswith((".zip", ".tar.gz", ".tgz", ".tar")))
    if not archives:
        print("No disc archives found in", DATA_DIR, "- contents:", [p.name for p in DATA_DIR.glob("*")])
    for path in archives:
        print("Processing:", path.name)
        if path.suffix.lower() == ".zip":
            with zipfile.ZipFile(path) as z:
                members = [m for m in z.infolist()
                           if "__MACOSX" not in Path(m.filename).parts
                           and not Path(m.filename).name.startswith("._")]
                z.extractall(MRI_ROOT, members=members)
        else:
            with tarfile.open(path, "r:*") as t:
                _safe_tar_extract(t, MRI_ROOT)
        print("Completed:", path.name)
print("Extraction step done.")

In [ ]:
from pathlib import Path
import re, random, copy
import numpy as np
import pandas as pd
import cv2
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, balanced_accuracy_score,
                             confusion_matrix, classification_report, roc_curve)

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


# Locate the clinical metadata (.xlsx/.csv): /content, Drive project folder, then extracted archives
def find_metadata():
    for folder in [WORK_DIR, DATA_DIR, MRI_ROOT]:
        if not folder.exists(): continue
        for pat in ['oasis_cross-sectional.xlsx','oasis_cross-sectional.csv']:
            hits=sorted(folder.glob(pat)) or sorted(folder.glob('**/'+pat))
            if hits: return str(hits[0])
    return None
METADATA_PATH=find_metadata()

MIN_AGE=60
FIXED_INDEX=80
CANDIDATES=range(50,111)  # RAS axial slice indices; confirm anatomy across participants
SIZE=128
BATCH=16
EPOCHS=35
PATIENCE=7

print('Using metadata file:', METADATA_PATH)
assert MRI_ROOT.exists(), f'Missing MRI folder: {MRI_ROOT}'
assert METADATA_PATH and Path(METADATA_PATH).exists(), (
    'Missing clinical metadata: put oasis_cross-sectional.xlsx (or .csv) in '
    f'{DATA_DIR} and re-run this cell.')

In [ ]:
print('Folder:', sorted(p.name for p in DATA_DIR.iterdir()))
print('T88 .img files extracted:', sum(1 for _ in MRI_ROOT.rglob('*t88_masked_gfc.img')))

In [ ]:
print('Metadata path in use:', METADATA_PATH)

## 2. Discover only real T88 `.img` files and match subject IDs

In [ ]:
paths=sorted(p for p in MRI_ROOT.rglob('*t88_masked_gfc.img')
             if '__MACOSX' not in p.parts and not p.name.startswith('._')
             and p.with_suffix('.hdr').exists())
rows=[]
for p in paths:
    m=re.search(r'OAS1_\d+',str(p))
    if m: rows.append({'subject_id':m.group(), 'mri_path':str(p)})
files_df=pd.DataFrame(rows)
assert len(files_df)>0,'No valid T88 files found; inspect your extraction paths.'
print('Volumes:',len(files_df),'Unique subjects:',files_df.subject_id.nunique())
print('Repeated scans:',files_df.subject_id.duplicated().sum())
display(files_df.head())

## 3. Clinical labels, eligibility, merge, and duplicate checks

In [ ]:
p=str(METADATA_PATH)
clinical=pd.read_csv(p) if p.lower().endswith('.csv') else pd.read_excel(p)
clinical.columns=clinical.columns.astype(str).str.strip()
print('Metadata columns:',clinical.columns.tolist())
assert {'ID','Age','CDR'}.issubset(clinical.columns)
clinical['subject_id']=clinical['ID'].astype(str).str.extract(r'(OAS1_\d+)',expand=False)
clinical['Age']=pd.to_numeric(clinical['Age'],errors='coerce')
clinical['CDR']=pd.to_numeric(clinical['CDR'],errors='coerce')
clinical=clinical.dropna(subset=['subject_id','Age','CDR']).copy()
clinical=clinical.loc[clinical.Age>=MIN_AGE].copy()
clinical['label']=(clinical.CDR>0).astype(int)
assert clinical.groupby('subject_id')['label'].nunique().max()==1, 'Conflicting labels: inspect repeated sessions'
# Deterministic selection; review scanner/session consistency if multiple scans available
files_df=files_df.sort_values(['subject_id','mri_path']).drop_duplicates('subject_id')
subjects=clinical.sort_values(['subject_id','ID']).drop_duplicates('subject_id')
data=subjects[['subject_id','Age','CDR','label']].merge(files_df,on='subject_id',validate='one_to_one')
print('Matched eligible subjects:',len(data))
print('Class counts (0=CDR0,1=CDR>0):',data.label.value_counts().to_dict())
assert set(data.label)=={0,1},'Need both classes; obtain additional MRI archives if required.'
assert data.label.value_counts().min() >= 8, 'Too few subjects in one class for stable train/val/test stratification.'
display(data.head())

## 4. Single subject-level split for BOTH methods

In [ ]:
train_df,tmp_df=train_test_split(data,test_size=.30,random_state=SEED,stratify=data.label)
val_df,test_df=train_test_split(tmp_df,test_size=.50,random_state=SEED,stratify=tmp_df.label)
sets=[set(x.subject_id) for x in (train_df,val_df,test_df)]
assert all(sets[i].isdisjoint(sets[j]) for i in range(3) for j in range(i+1,3))
for name,frame in [('train',train_df),('val',val_df),('test',test_df)]:
    print(name,len(frame),frame.label.value_counts().to_dict())
    frame.to_csv(str(WORK_DIR/f'oasis1_{name}_subjects.csv'),index=False)


## 5. Slice preprocessing and label-independent Canny selection

In [ ]:
def load_volume(path):
    img=nib.as_closest_canonical(nib.load(path))
    vol=np.squeeze(np.asarray(img.dataobj,dtype=np.float32))
    if vol.ndim!=3: raise ValueError(f'Expected 3D volume, got {vol.shape}: {path}')
    return vol

def normalize_uint8(s):
    s=np.nan_to_num(np.asarray(s,dtype=np.float32),nan=0,posinf=0,neginf=0)
    fg=s[s>0]
    if fg.size<100: return np.zeros(s.shape,dtype=np.uint8)
    lo,hi=np.percentile(fg,[1,99])
    z=np.clip((s-lo)/max(hi-lo,1e-6),0,1)
    return np.uint8(z*255)

def to_cnn_slice(vol,idx):
    # RAS axial orientation and same 128x128 transformations for both arms
    return cv2.resize(normalize_uint8(vol[:,:,idx].T),(SIZE,SIZE),interpolation=cv2.INTER_AREA)

def select_canny(vol):
    scores={}
    for idx in CANDIDATES:
        if idx>=vol.shape[2]: continue
        s=to_cnn_slice(vol,idx)
        blur=cv2.GaussianBlur(s,(5,5),1)
        edges=cv2.Canny(blur,50,150)
        scores[idx]=int(np.count_nonzero(edges))
    if not scores: raise ValueError('No available slices in CANDIDATES')
    # Consistent deterministic tie breaking: smallest index
    best=max(scores,key=lambda idx:(scores[idx],-idx))
    return best,scores

# Sanity-check one matched volume
v=load_volume(data.iloc[0].mri_path)
print('Volume shape:',v.shape)
assert v.shape[2]>FIXED_INDEX
idx,scores=select_canny(v)
print('Fixed slice:',FIXED_INDEX,'Canny slice:',idx)
fig,ax=plt.subplots(1,2,figsize=(8,4))
for a,j,title in zip(ax,[FIXED_INDEX,idx],['Fixed','Canny']):
    a.imshow(to_cnn_slice(v,j),cmap='gray');a.set_title(f'{title} slice {j}');a.axis('off')
plt.tight_layout();plt.show()

## 6. Precompute both sets of one-slice images (small memory footprint)

In [ ]:
fixed_cache={}; canny_cache={}; canny_indices={}; failures=[]
for k,row in enumerate(data.itertuples(index=False)):
    try:
        v=load_volume(row.mri_path)
        if FIXED_INDEX>=v.shape[2]: raise ValueError('Fixed slice out of range')
        ci,_=select_canny(v)
        fixed_cache[row.subject_id]=to_cnn_slice(v,FIXED_INDEX)
        canny_cache[row.subject_id]=to_cnn_slice(v,ci)
        canny_indices[row.subject_id]=ci
    except Exception as exc:
        failures.append((row.subject_id,str(exc)))
print('Usable:',len(fixed_cache),'Failed:',len(failures))
print('Failures (first 10):',failures[:10])
# Exclude subjects that failed from BOTH arms, preserving preassigned partitions
usable=set(fixed_cache)
train_df=train_df[train_df.subject_id.isin(usable)].copy()
val_df=val_df[val_df.subject_id.isin(usable)].copy()
test_df=test_df[test_df.subject_id.isin(usable)].copy()
assert all(set(x.label)=={0,1} for x in (train_df,val_df,test_df)), 'After exclusions one split lost a class.'
print('Final N:',[len(x) for x in (train_df,val_df,test_df)])
plt.figure(figsize=(7,3));plt.hist(list(canny_indices.values()),bins=25)
plt.xlabel('Canny-selected axial index');plt.ylabel('Number of subjects');plt.title('Slice selection distribution');plt.show()


## 7. Dataset and DataLoaders — identical transforms and splits

In [ ]:
class SlicesDataset(Dataset):
    def __init__(self,frame,cache,augment=False):
        self.df=frame.reset_index(drop=True);self.cache=cache
        self.tf=transforms.Compose([
            transforms.ToPILImage(),
            *( [transforms.RandomRotation(5)] if augment else [] ),
            transforms.ToTensor(),
            transforms.Normalize((0.5,),(0.5,))
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        return self.tf(self.cache[row.subject_id]),torch.tensor(int(row.label),dtype=torch.long)

def loaders(cache):
    train=DataLoader(SlicesDataset(train_df,cache,True),batch_size=BATCH,shuffle=True,num_workers=0)
    val=DataLoader(SlicesDataset(val_df,cache),batch_size=BATCH,shuffle=False,num_workers=0)
    test=DataLoader(SlicesDataset(test_df,cache),batch_size=BATCH,shuffle=False,num_workers=0)
    return train,val,test
print('Batch shape:',next(iter(loaders(fixed_cache)[0]))[0].shape)

## 8. Same compact 2D CNN in both arms

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(64,32),nn.ReLU(),nn.Dropout(.3),nn.Linear(32,2))
    def forward(self,x): return self.net(x)

def train_one(name,cache):
    # Restore same initialization seed for both comparisons
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    model=SimpleCNN().to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
    criterion=nn.CrossEntropyLoss()
    dl_train,dl_val,dl_test=loaders(cache)
    best_loss=float('inf');best_state=None;wait=0
    history=[]
    for epoch in range(EPOCHS):
        model.train(); losses=[]
        for x,y in dl_train:
            x,y=x.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad()
            loss=criterion(model(x),y);loss.backward();optimizer.step()
            losses.append(loss.item()*len(y))
        train_loss=sum(losses)/len(dl_train.dataset)
        model.eval();loss_sum=0;correct=0
        with torch.no_grad():
            for x,y in dl_val:
                x,y=x.to(DEVICE),y.to(DEVICE)
                logits=model(x);loss_sum+=criterion(logits,y).item()*len(y)
                correct+=(logits.argmax(1)==y).sum().item()
        val_loss=loss_sum/len(dl_val.dataset)
        history.append((train_loss,val_loss,correct/len(dl_val.dataset)))
        print(f'{name}: epoch {epoch+1:02d} train_loss={train_loss:.3f} val_loss={val_loss:.3f} val_acc={history[-1][2]:.3f}')
        if val_loss<best_loss-1e-4:
            best_loss=val_loss;best_state=copy.deepcopy(model.state_dict());wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    return model,np.array(history),dl_test

# Run both experiments; independent models, identical hyperparameters
fixed_model,fixed_hist,fixed_test=train_one('Fixed',fixed_cache)
canny_model,canny_hist,canny_test=train_one('Canny',canny_cache)

## 9. Evaluate ONCE on the untouched test set; compare side by side

In [ ]:
def evaluate(model,dl):
    model.eval();ys=[];ps=[]
    with torch.no_grad():
        for x,y in dl:
            p=model(x.to(DEVICE)).softmax(dim=1)[:,1].cpu().numpy()
            ys.extend(y.numpy());ps.extend(p)
    y=np.array(ys); p=np.array(ps); pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    metrics={'Accuracy':float(np.mean(pred==y)),
             'Balanced accuracy':balanced_accuracy_score(y,pred),
             'F1':f1_score(y,pred,zero_division=0),
             'Sensitivity':tp/(tp+fn) if tp+fn else np.nan,
             'Specificity':tn/(tn+fp) if tn+fp else np.nan,
             'ROC-AUC':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan}
    return metrics,y,p,pred

fm,fy,fp,fpred=evaluate(fixed_model,fixed_test)
cm,cy,cp,cpred=evaluate(canny_model,canny_test)
assert np.array_equal(fy,cy),'Test sample alignment differs!'
summary=pd.DataFrame([fm,cm],index=['Fixed slice 80','Canny selected']).T
print(summary.round(3).to_string())
print('\nFixed slice classification report:\n',classification_report(fy,fpred,zero_division=0))
print('Canny classification report:\n',classification_report(cy,cpred,zero_division=0))
fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].plot(fixed_hist[:,0],label='Fixed train');ax[0].plot(fixed_hist[:,1],label='Fixed val')
ax[0].plot(canny_hist[:,0],label='Canny train');ax[0].plot(canny_hist[:,1],label='Canny val')
ax[0].set_title('Learning curves');ax[0].set_xlabel('Epoch');ax[0].set_ylabel('Loss');ax[0].legend()
for a,y,pred,title in [(ax[1],fy,fpred,'Fixed'),(ax[2],cy,cpred,'Canny')]:
    mat=confusion_matrix(y,pred,labels=[0,1]);a.imshow(mat,cmap='Blues')
    a.set_title(title+' confusion matrix');a.set_xlabel('Predicted');a.set_ylabel('Actual')
    a.set_xticks([0,1]);a.set_yticks([0,1])
    for (i,j),v in np.ndenumerate(mat):a.text(j,i,str(v),ha='center',va='center')
plt.tight_layout();plt.show()
plt.figure(figsize=(6,5))
for label,y,p in [('Fixed',fy,fp),('Canny',cy,cp)]:
    fpr,tpr,_=roc_curve(y,p);plt.plot(fpr,tpr,label=f'{label}: AUC={roc_auc_score(y,p):.3f}')
plt.plot([0,1],[0,1],'--',color='gray');plt.xlabel('False positive rate');plt.ylabel('True positive rate');plt.legend();plt.title('Subject-level ROC curves');plt.show()

## 10. Save results and weights (optional Drive copy)

In [ ]:
OUT=WORK_DIR/'oasis1_comparison';OUT.mkdir(parents=True,exist_ok=True)
summary.to_csv(OUT/'test_metrics.csv')
pd.DataFrame({'subject_id':test_df.subject_id.to_numpy(),
              'true_label':fy,'fixed_probability':fp,'canny_probability':cp,
              'canny_slice_index':[canny_indices[s] for s in test_df.subject_id]}).to_csv(OUT/'test_predictions.csv',index=False)
torch.save(fixed_model.state_dict(),OUT/'fixed_cnn.pt')
torch.save(canny_model.state_dict(),OUT/'canny_cnn.pt')
print('Saved to:',OUT)
print('NOTE: results are in the folder printed above (on Colab, copy it to Drive before the runtime resets).')

In [ ]:
for name, frame in [("train_df", train_df), ("val_df", val_df), ("test_df", test_df)]:
    print(f"\n{name}")
    print("Total records:", len(frame))
    print("Unique subjects:", frame["subject_id"].nunique())
    print(frame["label"].value_counts())

## Interpretation and limitations

- **Canny is not a biomarker detector**: the most edge-dense slice might reflect brain size, image acquisition, noise, and preprocessing instead of disease-specific anatomy.
- Slice `80` and candidate indices `50..110` are **starting choices**, not validated anatomically optimal locations. Check matching anatomical levels in all volumes and keep the rule fixed on the held-out test set.
- Both models have one observation per subject. The modest older OASIS-1 sample means metrics have wide uncertainty. Consider repeated subject-level stratified cross-validation in future work; never choose the winner based on repeated peeks at the test set.
- For more rigor, assess age, sex and scan quality differences between the two classes; OASIS-1 CDR status is not pathologically confirmed Alzheimer's disease.
- Compare the Canny slice index histogram against the fixed index; many similar selections may explain similar model results.
